Got it. We’re framing this at a **production/portfolio-ready, executive-level analysis perspective**. SQL handled **raw aggregations and foundational metrics**, but Pandas is where you add **multi-season trends, comparative insights, and derived intelligence that SQL alone cannot easily express**, especially with rolling windows, consistency, and peer-relative metrics.

Here’s how we can define the **project problem and the executive-level questions** your Pandas analysis will answer:

---

### **Project Problem Statement**

“While SQL provides the historical metrics of drivers, constructors, and circuits in isolation, executives need actionable insights that account for trends, consistency, relative performance, and reliability across multiple seasons. These insights cannot be efficiently extracted using SQL alone because they require rolling calculations, multi-year aggregations, and comparative scoring that incorporate context and smoothing over time. The goal is to transform historical F1 race data into executive-ready insights to guide decisions on driver retention, constructor investment, and circuit prioritization.”

---

### **Executive-Level Questions to Answer in Pandas**

**1. Time-Based Performance Trends (Drivers & Constructors)**

* How have top drivers’ win rates, podium rates, and points per race evolved season-to-season?
* Which constructors show improvement or decline in team performance over rolling 3–5 year windows?
* Which circuits have become more or less challenging over time (DNF trends, win variability)?

**Why Pandas:** Rolling averages and multi-season trend lines are cumbersome in SQL. Pandas allows window-based smoothing, multi-group rolling metrics, and fast experimentation with different time windows.

---

**2. Comparative Performance Analysis**

* Which drivers consistently outperform their peers within the same season and across seasons?
* How do constructors compare when accounting for driver DNFs and point accumulation?
* Which circuits are the most reliable or risky relative to other circuits?

**Why Pandas:** Percentile ranks, season-over-season ranking, and comparative top-N selection are computationally trivial in Pandas but require nested subqueries and complex CTEs in SQL. Pandas allows a fluid, production-level approach for executive dashboards.

---

**3. Consistency & Reliability Metrics**

* Which drivers maintain stable performance across multiple seasons?
* Which constructors have stable points accumulation despite variable driver outcomes?
* Which circuits consistently produce high variability or failures?

**Why Pandas:** SQL cannot natively compute rolling standard deviations, stability scores, or multi-metric consistency indices efficiently. Pandas allows building **custom performance consistency and volatility metrics** for executive insights.

---

**4. Failure & Risk Insights**

* Which drivers or constructors are most affected by DNFs, and what is the nature of those failures (driver vs mechanical)?
* Which circuits produce the highest risk of DNFs or inconsistent outcomes?
* How do DNFs impact overall championship performance trends for drivers and teams?

**Why Pandas:** Calculating rolling DNF rates, correlating DNFs with points lost, or categorizing failures is cumbersome in SQL but straightforward in Pandas with `.groupby()`, `.rolling()`, and conditional logic. This produces **actionable insights for risk management and decision-making**.

---

**Summary:**

* **SQL:** foundational metrics, aggregations, and master table creation.
* **Pandas:** trend analysis, comparative insights, consistency and reliability scoring, percentile rankings, risk profiling.
* **Tableau:** executive storytelling, visualization, dashboards for these insights.

This framing clearly justifies every choice in your workflow and positions your analysis as **production-level, executive-ready, and decision-oriented**.

If you want, I can next **convert this into a concise “Pandas Analysis Problem & Questions” section** that would fit directly in your README or project documentation — the kind recruiters/readers immediately understand your reasoning and analytical depth.

Do you want me to do that?


Step 1: Data Preparation / Validation
Goal: Clean, join, and validate the dataset to ensure trustworthy metrics.

1. Inspect with intent (not just select *)
For each table, answer:
	•	What is the primary key?
	•	What columns are actually needed for analysis?
	•	What columns are dirty or irrelevant?
Example thinking:
	•	results → core fact table (KEEP most)
	•	drivers → only name + id (DROP rest)
	•	constructors → name + id
	•	races → raceId, year, round, circuitId
	•	circuits → name, location

2. Build clean CTEs (not raw copies)
Each CTE should:
	•	Select only required columns
	•	Rename for clarity
	•	Cast types if needed
If your CTE still looks like select *, you’re doing it wrong.

3. Define the grain explicitly (most important)
You said:
“one driver per race per season”
Correct. Lock this mentally:
Grain = one driver in one race
Everything you join must respect this.

4. Join order (this is where people mess up)
Correct flow:
	1	Start with results (fact table)
	2	Join drivers (driverId)
	3	Join constructors (constructorId)
	4	Join races (raceId)
	5	Join circuits (via race)
If you join in random order, you risk duplication.

5. Validate after join (mandatory)
After creating master table, check:
	•	Row count = same as results
	•	No duplicate (driverId, raceId)
	•	Nulls in critical columns?
If this fails → your joins are wrong.

	•	SQL / BigQuery:
	◦	Join results_df with drivers_df, constructors_df, races_df, circuits_df.
	◦	Check for missing DNFs, duplicate entries, and invalid points.
	◦	Create intermediate tables: driver_results, constructor_results, race_summary.
	◦	Use CTEs and window functions to compute cumulative stats per driver/constructor.
	•	Python / Pandas:
	◦	Validate data consistency: duplicated(), isnull(), merge checks.
	◦	Create helper columns: season, age, rookie flag, race order.
Deliverable: Clean, normalized tables ready for analysis.

Step 2: Driver Performance & True Skill
SQL / BigQuery:
	•	Compute win rate, podium rate, points per race, DNF-adjusted points.
	•	Use window functions: RANK() over season per driver, LAG() to compute performance deltas.
	•	Compute driver vs teammate delta: points_this_race - teammate_points_this_race.
Python / Pandas:
	•	Compute rolling averages, trends over seasons, rookie vs veteran comparisons.
	•	Normalize metrics across eras or points system changes.
Looker Studio:
	•	KPI tiles: win rate, podium %, teammate delta.
	•	Trend charts: career trajectory, rookie growth, top drivers over seasons.
Insights: Highlight overperformers, consistent drivers, and risk-prone drivers (high DNF).

Step 3: Team / Constructor Analysis
SQL / BigQuery:
	•	Aggregate points per constructor, points per driver, win contribution ratios.
	•	Identify dependency on top driver via SUM(points)/SUM(total_points).
	•	Analyze DNFs by constructor and circuit.
Python:
	•	Compute variance metrics: driver contribution distribution.
	•	Operational efficiency: average points lost per race due to DNF or pit stop delays.
Looker Studio:
	•	Stacked bar charts: driver contribution per team.
	•	Scatter plots: reliability vs points per team.
Insights: Identify operational risks, driver dependency, and efficiency.

Step 4: Circuit & Race Analysis
SQL / BigQuery:
	•	Compute circuit-wise average winner dominance, number of unique winners, race unpredictability.
	•	Use window functions for track-based rankings, e.g., most frequent podiums per driver.
Python:
	•	Track variance, historical trends, impact of track type (street vs permanent).
Looker Studio:
	•	Heatmaps: circuits vs driver success rate.
	•	Charts: most unpredictable tracks, DNF-prone circuits.
Insights: Recommend circuits for competitive parity or highlight marketing storylines.

Step 5: Career & Talent Trajectory
SQL / BigQuery:
	•	Compute career points per season, peak season, rolling 3-year averages.
Python / Pandas:
	•	Identify rookies outperforming historical trends.
	•	Visualize age-performance curves.
Looker Studio:
	•	Line charts: driver career trajectories, peak performance age.
Insights: Scout emerging talent and guide retention/contract strategy.

Step 6: Competitive Balance
SQL / BigQuery:
	•	Compute concentration metrics: Gini coefficient of points, top driver/constructor share.
Python:
	•	Cross-era normalization, trend analysis.
Looker Studio:
	•	KPI tiles: % points held by top 3 drivers/teams per season.
	•	Line charts: competitive balance over decades.
Insights: Suggest league health interventions, dominance monitoring.

Step 7: Documentation & Presentation
	•	Document methodology: tables, calculations, assumptions.
	•	Explain insights per metric: metric → business impact → executive recommendation.
	•	Slides: Visuals + concise insights, no raw queries.
	•	Appendix: SQL snippets, Python derivations, data checks.


26 march 2026

My base table is results. All joins must preserve its row count and grain.

1. Objective (always first)
Build a master dataset at driver-race level to enable performance, team, and circuit analysis.”

1. Objective (always first)
Table: results
Grain: one driver per race
Primary key: (raceId, driverId)
Purpose: core performance metrics (position, points, grid, status)

results is the fact table. It defines performance at driver-race level. All metrics (win rate, points, DNF, consistency) originate from this table.
positionText is inconsistent and not used for logic. Race completion status will be derived 
from statusId via the status table.

Status is race status: finished, and def.
“Race outcomes will be classified into Finished, Mechanical Failure, Driver Error, and Disqualification to enable reliability and risk analysis.”

DNF and outcome classification will be applied in the final joined table, after integrating status with results.

drivers table enriches performance data with identity and enables career/age-based analysis.



All joins will be performed using surrogate keys (driverId, constructorId, raceId). Text references like driverRef are excluded to avoid inconsistency.”
————————
Drivers to be cleaned, handle null values and select columns

drivers table enriches performance data with identity and enables career/age-based analysis

constructors table enables team-level performance and dependency analysis.

races table provides temporal and event context, enabling season-level and circuit-level performance insights.


circuits table provides geographic and track-level context for performance and competitiveness analysis.



SQL layer will produce clean, reusable base metrics. Pandas layer will focus on advanced analytics and insight generation. No duplication of logic across layers.

3. Join Strategy (this is critical)
Join drivers on driverId
Join constructors on constructorId
Join races on raceId
Join circuits via races.circuitId

Data Validation Checks
Row count (results): X  
Row count (final): X → PASS  
Duplicate (driverId, raceId): 0 → PASS  

Execution model:

Join Order:
1. results + drivers (driver identity)
2. + constructors (team context)
3. + races (time context)
4. + circuits (location context)
5. + status (outcome classification)

5. Assumptions / Decisions
DNF defined as status != finished”
“Only numeric positions considered”
“Dropped unnecessary columns to reduce noise”




In [2]:
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project = 'project-1-music-489315')
print("Connected")

/Users/Anirudh/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/Anirudh/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/Anirudh/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version,

Connected


In [3]:
results_master_df = client.query("SELECT * FROM `project-1-music-489315.Formula_1.results_master`").to_dataframe()
results_master_df.head()

/Users/Anirudh/Library/Python/3.9/lib/python/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,grid,position,points,laps,fastestLap,driverId,driver_name,dob,driver_nationality,constructorId,...,circuitId,circuit_name,circuit_city,circuit_country,lat,lng,alt,statusId,race_status,failure_type
0,3,<NA>,0.0,62,\N,579,Juan Fangio,1911-06-24,Argentine,51,...,9,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153,44,is_dnf,Mechanical / reliability failure
1,7,<NA>,0.0,8,\N,789,Eugène Martin,1915-03-24,French,154,...,9,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153,51,is_dnf,Mechanical / reliability failure
2,17,<NA>,0.0,43,\N,785,Geoff Crossley,1921-05-11,British,126,...,9,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153,7,is_dnf,Mechanical / reliability failure
3,18,<NA>,0.0,44,\N,747,David Murray,1909-12-28,British,105,...,9,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153,5,is_dnf,Mechanical / reliability failure
4,12,<NA>,0.0,2,\N,790,Leslie Johnson,1912-03-22,British,151,...,9,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153,126,is_dnf,Mechanical / reliability failure


In [4]:
results_master_df.columns

Index(['grid', 'position', 'points', 'laps', 'fastestLap', 'driverId',
       'driver_name', 'dob', 'driver_nationality', 'constructorId',
       'constructor_name', 'constructor_nationality', 'raceId', 'race_year',
       'race_round', 'race_name', 'race_date', 'circuitId', 'circuit_name',
       'circuit_city', 'circuit_country', 'lat', 'lng', 'alt', 'statusId',
       'race_status', 'failure_type'],
      dtype='object')

In [5]:
driver_metrics_df = client.query("SELECT * FROM `project-1-music-489315.Formula_1.driver_metrics`").to_dataframe()
driver_metrics_df.head()

/Users/Anirudh/Library/Python/3.9/lib/python/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,driver_name,total_races,total_points,total_wins,total_podiums,total_dnfs,points_per_race,win_rate,podium_rate,dnf_rate
0,Robert Kubica,99,274.0,1,12,30,2.77,0.01,0.12,0.30
1,Max Verstappen,209,2912.5,63,112,39,13.94,0.30,0.54,0.19
2,Fernando Alonso,404,2329.0,32,106,132,5.76,0.08,0.26,0.33
3,Roberto Moreno,74,15.0,0,1,66,0.20,0.00,0.01,0.89
4,Stefano Modena,81,17.0,0,2,61,0.21,0.00,0.02,0.75


In [6]:
constructor_metrics_df = client.query("SELECT * FROM `project-1-music-489315.Formula_1.constructor_metrics`").to_dataframe()
constructor_metrics_df.head()

/Users/Anirudh/Library/Python/3.9/lib/python/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,constructor_name,total_races,total_points,total_wins,total_podiums,total_dnfs,points_per_race,win_rate,podium_rate,dnf_rate
0,Tyrrell,881,711.0,23,77,573,0.81,0.03,0.09,0.65
1,Kurtis Kraft,226,130.0,5,19,118,0.58,0.02,0.08,0.52
2,Brabham,662,631.0,23,78,457,0.95,0.03,0.12,0.69
3,Lotus-Climax,231,281.0,22,31,144,1.22,0.10,0.13,0.62
4,March,524,148.0,3,16,384,0.28,0.01,0.03,0.73


In [7]:
circuit_metrics_df = client.query("SELECT * FROM `project-1-music-489315.Formula_1.circuit_metrics`").to_dataframe()
circuit_metrics_df.head()

/Users/Anirudh/Library/Python/3.9/lib/python/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,circuit_name,total_races,total_points,total_wins,total_podiums,total_dnfs,points_per_race,win_rate,podium_rate,dnf_rate
0,Autódromo Hermanos Rodríguez,558,1290.00,24,72,327,2.31,0.04,0.13,0.59
1,Suzuka Circuit,791,1923.00,34,102,436,2.43,0.04,0.13,0.55
2,Circuit de Spa-Francorchamps,1258,2591.50,57,172,689,2.06,0.05,0.14,0.55
3,Silverstone Circuit,1436,2798.56,59,178,800,1.95,0.04,0.12,0.56
4,Autódromo José Carlos Pace,937,2203.00,41,123,551,2.35,0.04,0.13,0.59


In [8]:
"""
Step 1: Driver Insights (pandas)

Focus on trend and comparative analysis:

Performance over seasons
Group by driver_name + year
Compute rolling win_rate, podium_rate, points_per_race
Identify improvement or decline trends
Consistency & reliability
Variance/std of finishing positions
Rolling DNF rate
Teammate comparisons
Merge driver_metrics with constructors per season
Compare driver vs teammate within same team
Step 2: Constructor Insights
Team dominance over time
Total points per season
Win/podium trends
Reliability
Mechanical vs driver-related failures
DNF distribution per team
Driver contribution
% of points contributed by each driver to constructor
Step 3: Circuit Insights
Circuit-specific difficulty
Average DNF per circuit
Average points per driver
Win rates per circuit
Location trends
Are certain drivers better in certain continents or climates?
"""

'\nStep 1: Driver Insights (pandas)\n\nFocus on trend and comparative analysis:\n\nPerformance over seasons\nGroup by driver_name + year\nCompute rolling win_rate, podium_rate, points_per_race\nIdentify improvement or decline trends\nConsistency & reliability\nVariance/std of finishing positions\nRolling DNF rate\nTeammate comparisons\nMerge driver_metrics with constructors per season\nCompare driver vs teammate within same team\nStep 2: Constructor Insights\nTeam dominance over time\nTotal points per season\nWin/podium trends\nReliability\nMechanical vs driver-related failures\nDNF distribution per team\nDriver contribution\n% of points contributed by each driver to constructor\nStep 3: Circuit Insights\nCircuit-specific difficulty\nAverage DNF per circuit\nAverage points per driver\nWin rates per circuit\nLocation trends\nAre certain drivers better in certain continents or climates?\n'

In [9]:
'''DRIVER TRENDS OVER SEASONS'''
driver_year = results_master_df.groupby(by=["driver_name", "race_year"]).agg(
    total_races = ("raceId", "count"),
    total_wins = ("position", lambda x: (x == 1).sum()),
    total_podiums = ("position", lambda x: (x <= 3).sum()),
    total_dnfs = ("race_status", lambda x: (x == "is_dnf").sum()),
    total_points = ("points", "sum")
).reset_index()

In [10]:
driver_year.head()

,driver_name,race_year,total_races,total_wins,total_podiums,total_dnfs,total_points
0,Adolf Brudes,1952,1,0,0,1,0.0
1,Adolfo Cruz,1953,1,0,0,1,0.0
2,Adrian Sutil,2007,17,0,0,10,1.0
3,Adrian Sutil,2008,18,0,0,14,0.0
4,Adrian Sutil,2009,17,0,0,11,5.0


In [11]:
'''Per year rates'''
driver_year["win_rate"] = driver_year["total_wins"] / driver_year["total_races"]
driver_year["podium_rate"] = driver_year["total_podiums"] / driver_year["total_races"]
driver_year["dnf_rate"] = driver_year["total_dnfs"] / driver_year["total_races"]
driver_year["points_per_race"] = driver_year["total_points"] / driver_year["total_races"]

In [12]:
driver_year.head()

,driver_name,race_year,total_races,total_wins,total_podiums,total_dnfs,total_points,win_rate,podium_rate,dnf_rate,points_per_race
0,Adolf Brudes,1952,1,0,0,1,0.0,0.0,0.0,1.0,0.0
1,Adolfo Cruz,1953,1,0,0,1,0.0,0.0,0.0,1.0,0.0
2,Adrian Sutil,2007,17,0,0,10,1.0,0.0,0.0,0.588235,0.058824
3,Adrian Sutil,2008,18,0,0,14,0.0,0.0,0.0,0.777778,0.0
4,Adrian Sutil,2009,17,0,0,11,5.0,0.0,0.0,0.647059,0.294118


In [13]:
'''Rolling metrics'''
driver_year = driver_year.sort_values(["driver_name", "race_year"])
driver_year["rolling_win_rate"] = driver_year.groupby("driver_name")["win_rate"].rolling(5, min_periods = 1).mean().reset_index(level = 0, drop = True)
driver_year["rolling_points"] = driver_year.groupby("driver_name")["points_per_race"].rolling(5, min_periods = 1).mean().reset_index(level = 0, drop = True)
driver_year["rolling_podium_rate"] = driver_year.groupby("driver_name")["podium_rate"].rolling(5, min_periods = 1).mean().reset_index(level = 0, drop = True)
driver_year["rolling_dnf_rate"] = driver_year.groupby("driver_name")["dnf_rate"].rolling(5, min_periods = 1).mean().reset_index(level = 0, drop = True)

In [14]:
driver_year.head()

,driver_name,race_year,total_races,total_wins,total_podiums,total_dnfs,total_points,win_rate,podium_rate,dnf_rate,points_per_race,rolling_win_rate,rolling_points,rolling_podium_rate,rolling_dnf_rate
0,Adolf Brudes,1952,1,0,0,1,0.0,0.0,0.0,1.0,0.0,0.0,0.000000,0.0,1.000000
1,Adolfo Cruz,1953,1,0,0,1,0.0,0.0,0.0,1.0,0.0,0.0,0.000000,0.0,1.000000
2,Adrian Sutil,2007,17,0,0,10,1.0,0.0,0.0,0.588235,0.058824,0.0,0.058824,0.0,0.588235
3,Adrian Sutil,2008,18,0,0,14,0.0,0.0,0.0,0.777778,0.0,0.0,0.029412,0.0,0.683007
4,Adrian Sutil,2009,17,0,0,11,5.0,0.0,0.0,0.647059,0.294118,0.0,0.117647,0.0,0.671024


In [15]:
'''CONSTRUCTOR TRENDS OVER SEASONS'''

constructor_year = results_master_df.groupby(["constructor_name", "race_year"]).agg(
    total_races = ("raceId", "count"),
    total_wins = ("position", lambda x : (x == 1).sum()),
    total_podiums = ("position", lambda x: (x <=3).sum()),
    total_dnfs = ("race_status", lambda x: (x == "is_dnf").sum()),
    total_points = ("points", "sum")
)

In [16]:
constructor_year.head()

total_races  total_wins  total_podiums  \
constructor_name race_year                                           
AFM              1952                 3           0              0   
                 1953                 4           0              0   
AGS              1986                 2           0              0   
                 1987                14           0              0   
                 1988                16           0              0   

                            total_dnfs  total_points  
constructor_name race_year                            
AFM              1952                2           0.0  
                 1953                3           0.0  
AGS              1986                2           0.0  
                 1987                5           1.0  
                 1988               13           0.0

In [17]:
'''Constructor Per Year Metrics'''
constructor_year["win_rate"] = constructor_year["total_wins"] / constructor_year["total_races"]
constructor_year["podium_rate"] = constructor_year["total_podiums"] / constructor_year["total_races"]
constructor_year["dnf_rate"] = constructor_year["total_dnfs"] / constructor_year["total_races"]
constructor_year["points_per_race"] = constructor_year["total_points"] / constructor_year["total_races"]

In [18]:
'''Rolling Metrics for Constructors'''
constructor_year = constructor_year.sort_values(["constructor_name", "race_year"])
constructor_year["rolling_win_rate"] = constructor_year.groupby("constructor_name")["win_rate"].rolling(5, min_periods = 1).mean().reset_index(level = 0, drop = True)
constructor_year["rolling_podium_rate"] = constructor_year.groupby("constructor_name")["podium_rate"].rolling(5, min_periods = 1).mean().reset_index(level = 0, drop = True)
constructor_year["rolling_dnf_rate"] = constructor_year.groupby("constructor_name")["dnf_rate"].rolling(5, min_periods = 1).mean().reset_index(level = 0, drop = True)
constructor_year["rolling_points"] = constructor_year.groupby("constructor_name")["points_per_race"].rolling(5, min_periods = 1).mean().reset_index(level = 0, drop = True)

In [19]:
constructor_year.head(10)

total_races  total_wins  total_podiums  \
constructor_name race_year                                           
AFM              1952                 3           0              0   
                 1953                 4           0              0   
AGS              1986                 2           0              0   
                 1987                14           0              0   
                 1988                16           0              0   
                 1989                31           0              0   
                 1990                32           0              0   
                 1991                28           0              0   
ATS              1963                17           0              0   
                 1978                32           0              0   

                            total_dnfs  total_points  win_rate  podium_rate  \
constructor_name race_year                                                    
AFM              1952                2           0.0       0.0          0.0   
                 1953                3           0.0       0.0          0.0   
AGS              1986                2           0.0       0.0          0.0   
                 1987                5           1.0       0.0          0.0   
                 1988               13           0.0       0.0          0.0   
                 1989               31           1.0       0.0          0.0   
                 1990               30           0.0       0.0          0.0   
                 1991               27           0.0       0.0          0.0   
ATS              1963               15           0.0       0.0          0.0   
                 1978               24           0.0       0.0          0.0   

                            dnf_rate  points_per_race  rolling_win_rate  \
constructor_name race_year                                                
AFM              1952       0.666667              0.0               0.0   
                 1953           0.75              0.0               0.0   
AGS              1986            1.0              0.0               0.0   
                 1987       0.357143         0.071429               0.0   
                 1988         0.8125              0.0               0.0   
                 1989            1.0         0.032258               0.0   
                 1990         0.9375              0.0               0.0   
                 1991       0.964286              0.0               0.0   
ATS              1963       0.882353              0.0               0.0   
                 1978           0.75              0.0               0.0   

                            rolling_podium_rate  rolling_dnf_rate  \
constructor_name race_year                                          
AFM              1952                       0.0          0.666667   
                 1953                       0.0          0.708333   
AGS              1986                       0.0          1.000000   
                 1987                       0.0          0.678571   
                 1988                       0.0          0.723214   
                 1989                       0.0          0.792411   
                 1990                       0.0          0.821429   
                 1991                       0.0          0.814286   
ATS              1963                       0.0          0.882353   
                 1978                       0.0          0.816176   

                            rolling_points  
constructor_name race_year                  
AFM              1952             0.000000  
                 1953             0.000000  
AGS              1986             0.000000  
                 1987             0.035714  
                 1988             0.023810  
                 1989             0.025922  
                 1990             0.020737  
                 1991             0.020737  
ATS              1963             0.000000  
             

In [20]:
# rounding off to 2 decimals
driver_year[["win_rate", "podium_rate", "dnf_rate",	"points_per_race", "rolling_win_rate", "rolling_points", "rolling_podium_rate", "rolling_dnf_rate"]] = driver_year[["win_rate", "podium_rate", "dnf_rate",	"points_per_race", "rolling_win_rate", "rolling_points", "rolling_podium_rate", "rolling_dnf_rate"]].round(2)
constructor_year[["win_rate",	"podium_rate",	"dnf_rate",	"points_per_race",	"rolling_win_rate",	"rolling_podium_rate",	"rolling_dnf_rate",	"rolling_points"]] = constructor_year[["win_rate",	"podium_rate",	"dnf_rate",	"points_per_race",	"rolling_win_rate",	"rolling_podium_rate",	"rolling_dnf_rate",	"rolling_points"]].round(2)

In [21]:
driver_year.head(10)

,driver_name,race_year,total_races,total_wins,total_podiums,total_dnfs,total_points,win_rate,podium_rate,dnf_rate,points_per_race,rolling_win_rate,rolling_points,rolling_podium_rate,rolling_dnf_rate
0,Adolf Brudes,1952,1,0,0,1,0.0,0.0,0.0,1.0,0.0,0.0,0.00,0.0,1.00
1,Adolfo Cruz,1953,1,0,0,1,0.0,0.0,0.0,1.0,0.0,0.0,0.00,0.0,1.00
2,Adrian Sutil,2007,17,0,0,10,1.0,0.0,0.0,0.59,0.06,0.0,0.06,0.0,0.59
3,Adrian Sutil,2008,18,0,0,14,0.0,0.0,0.0,0.78,0.0,0.0,0.03,0.0,0.68
4,Adrian Sutil,2009,17,0,0,11,5.0,0.0,0.0,0.65,0.29,0.0,0.12,0.0,0.67
5,Adrian Sutil,2010,19,0,0,7,47.0,0.0,0.0,0.37,2.47,0.0,0.71,0.0,0.60
6,Adrian Sutil,2011,19,0,0,12,42.0,0.0,0.0,0.63,2.21,0.0,1.01,0.0,0.60
7,Adrian Sutil,2013,19,0,0,10,29.0,0.0,0.0,0.53,1.53,0.0,1.30,0.0,0.59
8,Adrian Sutil,2014,19,0,0,17,0.0,0.0,0.0,0.89,0.0,0.0,1.30,0.0,0.61
9,Adrián Campos,1987,16,0,0,15,0.0,0.0,0.0,0.94,0.0,0.0,0.00,0.0,0.94


In [22]:
constructor_year.head(10)

total_races  total_wins  total_podiums  \
constructor_name race_year                                           
AFM              1952                 3           0              0   
                 1953                 4           0              0   
AGS              1986                 2           0              0   
                 1987                14           0              0   
                 1988                16           0              0   
                 1989                31           0              0   
                 1990                32           0              0   
                 1991                28           0              0   
ATS              1963                17           0              0   
                 1978                32           0              0   

                            total_dnfs  total_points  win_rate  podium_rate  \
constructor_name race_year                                                    
AFM              1952                2           0.0       0.0          0.0   
                 1953                3           0.0       0.0          0.0   
AGS              1986                2           0.0       0.0          0.0   
                 1987                5           1.0       0.0          0.0   
                 1988               13           0.0       0.0          0.0   
                 1989               31           1.0       0.0          0.0   
                 1990               30           0.0       0.0          0.0   
                 1991               27           0.0       0.0          0.0   
ATS              1963               15           0.0       0.0          0.0   
                 1978               24           0.0       0.0          0.0   

                            dnf_rate  points_per_race  rolling_win_rate  \
constructor_name race_year                                                
AFM              1952           0.67              0.0               0.0   
                 1953           0.75              0.0               0.0   
AGS              1986            1.0              0.0               0.0   
                 1987           0.36             0.07               0.0   
                 1988           0.81              0.0               0.0   
                 1989            1.0             0.03               0.0   
                 1990           0.94              0.0               0.0   
                 1991           0.96              0.0               0.0   
ATS              1963           0.88              0.0               0.0   
                 1978           0.75              0.0               0.0   

                            rolling_podium_rate  rolling_dnf_rate  \
constructor_name race_year                                          
AFM              1952                       0.0              0.67   
                 1953                       0.0              0.71   
AGS              1986                       0.0              1.00   
                 1987                       0.0              0.68   
                 1988                       0.0              0.72   
                 1989                       0.0              0.79   
                 1990                       0.0              0.82   
                 1991                       0.0              0.81   
ATS              1963                       0.0              0.88   
                 1978                       0.0              0.82   

                            rolling_points  
constructor_name race_year                  
AFM              1952                 0.00  
                 1953                 0.00  
AGS              1986                 0.00  
                 1987                 0.04  
                 1988                 0.02  
                 1989                 0.03  
                 1990                 0.02  
                 1991                 0.02  
ATS              1963                 0.00  
             

In [23]:
print(driver_year["race_year"].dtype)

Int64


In [24]:
print(driver_year["race_year"].head)

<bound method NDFrame.head of 0       1952
1       1953
2       2007
3       2008
4       2009
        ... 
3203    1991
3204    1992
3205    1993
3206    1994
3207    1956
Name: race_year, Length: 3208, dtype: Int64>


In [25]:
driver_year["race_year"] = pd.to_numeric(driver_year["race_year"], errors = "coerce")

In [26]:
print(driver_year["race_year"].dtype)
print(driver_year["race_year"].head)

Int64
<bound method NDFrame.head of 0       1952
1       1953
2       2007
3       2008
4       2009
        ... 
3203    1991
3204    1992
3205    1993
3206    1994
3207    1956
Name: race_year, Length: 3208, dtype: Int64>


In [27]:
# top 20 drivers by win rate and points per race
recent = driver_year[driver_year["race_year"] >= 2015]
top = recent.sort_values("rolling_win_rate", ascending = False).head(15)

In [28]:
top

,driver_name,race_year,total_races,total_wins,total_podiums,total_dnfs,total_points,win_rate,podium_rate,dnf_rate,points_per_race,rolling_win_rate,rolling_points,rolling_podium_rate,rolling_dnf_rate
1887,Lewis Hamilton,2020,16,11,14,0,347.0,0.69,0.88,0.0,21.69,0.53,19.41,0.79,0.04
1885,Lewis Hamilton,2018,21,11,17,1,408.0,0.52,0.81,0.05,19.43,0.51,19.19,0.80,0.08
1888,Lewis Hamilton,2021,22,8,17,1,385.5,0.36,0.77,0.05,17.52,0.51,19.29,0.78,0.03
2097,Max Verstappen,2024,24,9,14,1,399.0,0.38,0.58,0.04,16.62,0.50,18.13,0.76,0.11
1886,Lewis Hamilton,2019,21,11,17,0,413.0,0.52,0.81,0.0,19.67,0.50,19.08,0.79,0.05
2096,Max Verstappen,2023,22,19,21,0,530.0,0.86,0.95,0.0,24.09,0.45,17.45,0.72,0.12
1889,Lewis Hamilton,2022,22,0,9,3,233.0,0.0,0.41,0.14,10.59,0.42,17.78,0.74,0.05
1884,Lewis Hamilton,2017,20,9,13,1,363.0,0.45,0.65,0.05,18.15,0.42,17.29,0.69,0.09
1883,Lewis Hamilton,2016,21,10,17,2,380.0,0.48,0.81,0.1,18.1,0.37,15.56,0.63,0.14
2875,Sebastian Vettel,2015,19,3,13,2,278.0,0.16,0.68,0.11,14.63,0.33,15.80,0.63,0.09


Category 1: Driver Performance Evolution
Q1. Which drivers show sustained dominance vs short-term peaks?
Not “highest win rate”
But: rolling win_rate trajectory over 5 seasons

What you’re detecting:

Sustained elite performers (flat high curve)
One-season wonders (spikes, then drop)
Late bloomers (gradual rise)

Why Pandas:

Rolling smoothing across seasons
SQL cannot easily express temporal smoothing + continuity

Actionable insight:

Retain drivers with stable high rolling performance
Avoid drivers with volatile spikes

In [29]:
latest = driver_year.sort_values("race_year").groupby("driver_name").tail(1) # sort by race year ascending, then group by driver name. now, the output is race_year, driver name. out of these, we need latest race year, so tail(1) gives us the last year a particular driver had a race
latest = latest.merge(driver_metrics_df[["driver_name", "total_races"]].rename(columns = {"total_races": "career_races"}), on="driver_name") # 
latest = latest[latest["career_races"] >= 50]
top = latest.sort_values("rolling_win_rate", ascending = False).head(10)

In [30]:
top.head(20)

,driver_name,race_year,total_races,total_wins,total_podiums,total_dnfs,total_points,win_rate,podium_rate,dnf_rate,points_per_race,rolling_win_rate,rolling_points,rolling_podium_rate,rolling_dnf_rate,career_races
153,Max Verstappen,2024,24,9,14,1,399.0,0.38,0.58,0.04,16.62,0.50,18.13,0.76,0.11,209
9,Jim Clark,1968,1,1,1,0,9.0,1.0,1.0,0.0,9.0,0.49,4.73,0.53,0.47,73
0,Juan Fangio,1958,2,0,0,0,7.0,0.0,0.0,0.0,3.5,0.46,5.50,0.61,0.17,58
18,Jackie Stewart,1973,15,5,8,4,71.0,0.33,0.53,0.27,4.73,0.37,4.42,0.51,0.34,100
3,Stirling Moss,1961,9,2,2,5,21.0,0.22,0.22,0.56,2.33,0.32,3.18,0.39,0.48,73
72,Alain Prost,1993,16,7,12,4,99.0,0.44,0.75,0.25,6.19,0.29,4.93,0.64,0.27,202
80,Ayrton Senna,1994,3,0,0,3,0.0,0.0,0.0,1.0,0.0,0.26,3.71,0.46,0.49,162
88,Nigel Mansell,1995,2,0,0,1,0.0,0.0,0.0,0.5,0.0,0.24,3.36,0.38,0.45,192
99,Mika Häkkinen,2001,17,2,3,8,37.0,0.12,0.18,0.47,2.18,0.24,4.00,0.46,0.36,165
145,Lewis Hamilton,2024,24,2,5,2,207.0,0.08,0.21,0.08,8.62,0.23,13.66,0.51,0.07,356


In [31]:
# top 10 drivers with the highest rolling_win_rate (rolling over 5 seasons)
driver_year[driver_year["total_races"] > 10].sort_values(by="rolling_win_rate", ascending=False).drop_duplicates(subset="driver_name", keep = "first").head(10)

,driver_name,race_year,total_races,total_wins,total_podiums,total_dnfs,total_points,win_rate,podium_rate,dnf_rate,points_per_race,rolling_win_rate,rolling_points,rolling_podium_rate,rolling_dnf_rate
2119,Michael Schumacher,2004,18,13,15,2,148.0,0.72,0.83,0.11,8.22,0.56,7.22,0.77,0.14
1887,Lewis Hamilton,2020,16,11,14,0,347.0,0.69,0.88,0.0,21.69,0.53,19.41,0.79,0.04
2097,Max Verstappen,2024,24,9,14,1,399.0,0.38,0.58,0.04,16.62,0.50,18.13,0.76,0.11
1471,Jim Clark,1967,11,4,5,6,41.0,0.36,0.45,0.55,3.73,0.43,4.39,0.51,0.51
2873,Sebastian Vettel,2013,19,13,16,1,397.0,0.68,0.84,0.05,20.89,0.40,14.80,0.65,0.14
212,Ayrton Senna,1992,16,3,7,9,50.0,0.19,0.44,0.56,3.12,0.38,4.72,0.60,0.35
1300,Jackie Stewart,1973,15,5,8,4,71.0,0.33,0.53,0.27,4.73,0.37,4.42,0.51,0.34
1333,Jacques Villeneuve,1997,17,7,8,6,81.0,0.41,0.47,0.35,4.76,0.33,4.82,0.58,0.33
39,Alain Prost,1988,16,7,14,2,105.0,0.44,0.88,0.12,6.56,0.32,4.66,0.65,0.31
531,Damon Hill,1997,17,0,1,11,7.0,0.0,0.06,0.65,0.41,0.26,4.11,0.51,0.40


In [32]:
driver_year[driver_year["driver_name"].isin(["Michael Schumacher", "Lewis Hamilton", "Max Verstappen"])].sort_values("rolling_win_rate", ascending = False).head(10)

,driver_name,race_year,total_races,total_wins,total_podiums,total_dnfs,total_points,win_rate,podium_rate,dnf_rate,points_per_race,rolling_win_rate,rolling_points,rolling_podium_rate,rolling_dnf_rate
2119,Michael Schumacher,2004,18,13,15,2,148.0,0.72,0.83,0.11,8.22,0.56,7.22,0.77,0.14
1887,Lewis Hamilton,2020,16,11,14,0,347.0,0.69,0.88,0.0,21.69,0.53,19.41,0.79,0.04
1888,Lewis Hamilton,2021,22,8,17,1,385.5,0.36,0.77,0.05,17.52,0.51,19.29,0.78,0.03
1885,Lewis Hamilton,2018,21,11,17,1,408.0,0.52,0.81,0.05,19.43,0.51,19.19,0.80,0.08
1886,Lewis Hamilton,2019,21,11,17,0,413.0,0.52,0.81,0.0,19.67,0.50,19.08,0.79,0.05
2097,Max Verstappen,2024,24,9,14,1,399.0,0.38,0.58,0.04,16.62,0.50,18.13,0.76,0.11
2120,Michael Schumacher,2005,19,1,5,6,62.0,0.05,0.26,0.32,3.26,0.47,6.60,0.68,0.16
2118,Michael Schumacher,2003,16,6,8,4,93.0,0.38,0.5,0.25,5.81,0.46,6.45,0.73,0.18
2117,Michael Schumacher,2002,17,11,17,0,144.0,0.65,1.0,0.0,8.47,0.46,6.37,0.76,0.17
2096,Max Verstappen,2023,22,19,21,0,530.0,0.86,0.95,0.0,24.09,0.45,17.45,0.72,0.12


Q2. Which drivers are improving or declining over time?
Trend slope of rolling_points or rolling_win_rate

What you’re detecting:

Positive slope → emerging talent
Negative slope → decline/aging/competitive loss

### Sustained Driver Classification

In [33]:
driver_summary = driver_year.groupby("driver_name").agg(
    avg_roll_win = ("rolling_win_rate", "mean"),
    std_roll_win = ("rolling_win_rate", "std"),
    max_roll_win = ("rolling_win_rate", "max")
).reset_index()

In [46]:
driver_metrics_df.head()

,driver_name,total_races,total_points,total_wins,total_podiums,total_dnfs,points_per_race,win_rate,podium_rate,dnf_rate
0,Robert Kubica,99,274.0,1,12,30,2.77,0.01,0.12,0.30
1,Max Verstappen,209,2912.5,63,112,39,13.94,0.30,0.54,0.19
2,Fernando Alonso,404,2329.0,32,106,132,5.76,0.08,0.26,0.33
3,Roberto Moreno,74,15.0,0,1,66,0.20,0.00,0.01,0.89
4,Stefano Modena,81,17.0,0,2,61,0.21,0.00,0.02,0.75


In [48]:
driver_summary = driver_summary.merge(driver_metrics_df[["driver_name", "total_races"]].rename(columns = {"total_races": "career_races"}), on = "driver_name")
driver_summary = driver_summary[driver_summary["career_races"] >= 50]
driver_summary["avg_rank"] = driver_summary["avg_roll_win"].rank(pct=True)
driver_summary["std_rank"] = driver_summary["std_roll_win"].rank(pct=True, ascending = True)
driver_summary["peak_gap_rank"] = driver_summary["peak_gap"].rank(pct=True, ascending = True)

In [49]:
driver_summary.head()

,driver_name,avg_roll_win,std_roll_win,max_roll_win,peak_gap,avg_rank,std_rank,peak_gap_rank,sustained_score,score_rank,career_races
0,Adrian Sutil,0.000000,0.000000,0.00,0.000000,0.23125,0.23125,0.23125,0.424071,0.401554,128
1,Aguri Suzuki,0.000000,0.000000,0.00,0.000000,0.23125,0.23125,0.23125,0.424071,0.401554,88
2,Alain Prost,0.218462,0.098306,0.32,0.101538,0.97500,0.93750,0.89375,0.976910,0.970639,202
3,Alan Jones,0.082000,0.068280,0.17,0.088000,0.88750,0.88125,0.88125,0.960217,0.946459,117
4,Alessandro Nannini,0.006000,0.008944,0.02,0.014000,0.48750,0.51875,0.53125,0.857921,0.811744,77


In [50]:
driver_summary["sustained_score"] = (
    driver_summary["avg_rank"] * 0.5 +
    driver_summary["std_rank"] * 0.3 + 
    driver_summary["peak_gap_rank"] * 0.2
)

In [37]:

#top 10 drivers with highest avg roll win
top_10 = driver_summary.nlargest(n=10, columns="avg_roll_win")


In [38]:
# top percentile rank
driver_summary["avg_rank"] = driver_summary["avg_roll_win"].rank(pct=True)

In [39]:

driver_summary.sort_values(by="avg_rank", ascending=False).head()

,driver_name,avg_roll_win,std_roll_win,max_roll_win,peak_gap,avg_rank
474,Juan Fangio,0.422500,0.056252,0.48,0.057500,1.000000
522,Lewis Hamilton,0.319444,0.131349,0.53,0.210556,0.998837
577,Michael Schumacher,0.291579,0.163852,0.56,0.268421,0.997674
415,Jim Clark,0.262222,0.187735,0.49,0.227778,0.996512
16,Alberto Ascari,0.261667,0.164124,0.42,0.158333,0.995349


In [40]:
driver_summary["std_rank"] = driver_summary["std_roll_win"].rank(pct=True, ascending=True)
driver_summary["peak_gap_rank"] = driver_summary["peak_gap"].rank(pct=True, ascending=True)

In [41]:
driver_summary.head()

,driver_name,avg_roll_win,std_roll_win,max_roll_win,peak_gap,avg_rank,std_rank,peak_gap_rank
0,Adolf Brudes,0.0,NaN,0.0,0.0,0.433721,NaN,0.433721
1,Adolfo Cruz,0.0,NaN,0.0,0.0,0.433721,NaN,0.433721
2,Adrian Sutil,0.0,0.0,0.0,0.0,0.433721,0.401554,0.433721
3,Adrián Campos,0.0,0.0,0.0,0.0,0.433721,0.401554,0.433721
4,Aguri Suzuki,0.0,0.0,0.0,0.0,0.433721,0.401554,0.433721


In [42]:
# assigning weighted scores
driver_summary["sustained_score"] = (
    driver_summary["avg_rank"] * 0.5 +
    driver_summary["std_rank"] * 0.3 + 
    driver_summary["peak_gap_rank"] * 0.2
)

In [58]:
career_seasons = driver_year.groupby("driver_name")["race_year"].nunique().reset_index(name = "career_seasons")

In [59]:
career_seasons.head()

,driver_name,career_seasons
0,Adolf Brudes,1
1,Adolfo Cruz,1
2,Adrian Sutil,7
3,Adrián Campos,2
4,Aguri Suzuki,8


In [61]:
driver_summary = driver_summary.merge(career_seasons, on = "driver_name", how="left")

In [66]:
driver_summary[["driver_name", "career_seasons"]].sort_values("career_seasons", ascending = False).head()

,driver_name,career_seasons
38,Fernando Alonso,21
102,Michael Schumacher,19
87,Kimi Räikkönen,19
140,Rubens Barrichello,19
69,Jenson Button,18


In [67]:
driver_summary = driver_summary[driver_summary["career_seasons"] >= 5]

In [69]:
driver_summary[["driver_name", "career_seasons"]].sort_values("career_seasons", ascending = False).head()

,driver_name,career_seasons
38,Fernando Alonso,21
140,Rubens Barrichello,19
102,Michael Schumacher,19
87,Kimi Räikkönen,19
90,Lewis Hamilton,18


In [51]:
driver_summary.sort_values("sustained_score", ascending=False).head(15)

,driver_name,avg_roll_win,std_roll_win,max_roll_win,peak_gap,avg_rank,std_rank,peak_gap_rank,sustained_score,score_rank,career_races
102,Michael Schumacher,0.291579,0.163852,0.56,0.268421,0.987500,0.98750,0.99375,0.988750,0.994819,308
70,Jim Clark,0.262222,0.187735,0.49,0.227778,0.981250,1.00000,0.98750,0.988125,0.996546,73
90,Lewis Hamilton,0.319444,0.131349,0.53,0.210556,0.993750,0.96875,0.96875,0.981250,0.991364,356
101,Max Verstappen,0.174000,0.181059,0.50,0.326000,0.950000,0.99375,1.00000,0.973125,0.987910,209
11,Ayrton Senna,0.209091,0.133900,0.38,0.170909,0.968750,0.97500,0.94375,0.965625,0.984456,162
142,Sebastian Vettel,0.178750,0.118371,0.40,0.221250,0.956250,0.95625,0.97500,0.960000,0.981002,300
146,Stirling Moss,0.130000,0.138564,0.32,0.190000,0.934375,0.98125,0.96250,0.954062,0.982729,73
57,Jackie Stewart,0.207778,0.113444,0.37,0.162222,0.962500,0.95000,0.93750,0.953750,0.975820,100
2,Alain Prost,0.218462,0.098306,0.32,0.101538,0.975000,0.93750,0.89375,0.947500,0.970639,202
60,Jacques Villeneuve,0.107273,0.120589,0.33,0.222727,0.912500,0.96250,0.98125,0.941250,0.974093,165


In [43]:
# selecting top drivers top 10 percentile
driver_summary["score_rank"] = driver_summary["sustained_score"].rank(pct=True)
sustained_drivers = driver_summary[driver_summary["score_rank"] >= 0.9]

In [44]:
sustained_drivers.head(10)

,driver_name,avg_roll_win,std_roll_win,max_roll_win,peak_gap,avg_rank,std_rank,peak_gap_rank,sustained_score,score_rank
8,Alain Prost,0.218462,0.098306,0.32,0.101538,0.989535,0.963731,0.965116,0.976910,0.970639
11,Alan Jones,0.082000,0.068280,0.17,0.088000,0.967442,0.946459,0.962791,0.960217,0.946459
16,Alberto Ascari,0.261667,0.164124,0.42,0.158333,0.995349,0.989637,0.976744,0.989914,0.986183
62,Ayrton Senna,0.209091,0.133900,0.38,0.170909,0.988372,0.982729,0.980233,0.985051,0.984456
83,Bill Vukovich,0.246000,0.232551,0.50,0.254000,0.991860,0.996546,0.995349,0.993964,0.993092
94,Bob Sweikert,0.090000,0.124499,0.25,0.160000,0.969767,0.977547,0.977907,0.973729,0.968912
114,Bruce McLaren,0.041538,0.032621,0.09,0.048462,0.940698,0.899827,0.933721,0.927041,0.903282
127,Carlos Reutemann,0.066364,0.037489,0.12,0.053636,0.960465,0.910190,0.940698,0.941429,0.927461
154,Clay Regazzoni,0.046364,0.025796,0.12,0.073636,0.944186,0.884283,0.953488,0.928076,0.906736
168,Damon Hill,0.173750,0.089752,0.26,0.086250,0.983721,0.960276,0.961628,0.972269,0.965458


Category 2: Constructor Performance Trends

Q3. Which constructors are building long-term dominance vs short bursts?
Rolling team win_rate / points_per_race

What you’re detecting:

Dynasties (consistent high rolling metrics)
Volatile teams (high variance)
Rebuilding teams (gradual improvement)

Actionable insight:

Long-term sponsors prefer stable teams

Short bursts = risky investment

In [70]:
constructor_metrics_df.describe()

,total_races,total_points,total_wins,total_podiums,total_dnfs,points_per_race,win_rate,podium_rate,dnf_rate
count,83.0,83.000000,83.0,83.0,83.0,83.000000,83.000000,83.000000,83.000000
mean,306.975904,632.946386,13.289157,40.168675,177.614458,1.076386,0.020361,0.068193,0.644578
std,405.420678,1851.832153,40.231807,118.956112,196.186188,1.857642,0.038301,0.093537,0.153871
min,52.0,0.000000,0.0,0.0,33.0,0.000000,0.000000,0.000000,0.160000
25%,81.5,16.000000,0.0,0.0,53.5,0.115000,0.000000,0.000000,0.555000
50%,154.0,59.000000,0.0,5.0,106.0,0.500000,0.000000,0.020000,0.660000
75%,402.0,310.570000,4.0,18.0,211.5,1.155000,0.020000,0.100000,0.735000
max,2439.0,11091.270000,249.0,841.0,969.0,11.860000,0.200000,0.460000,0.950000


In [71]:
constructor_metrics_df.head()

,constructor_name,total_races,total_points,total_wins,total_podiums,total_dnfs,points_per_race,win_rate,podium_rate,dnf_rate
0,Tyrrell,881,711.0,23,77,573,0.81,0.03,0.09,0.65
1,Kurtis Kraft,226,130.0,5,19,118,0.58,0.02,0.08,0.52
2,Brabham,662,631.0,23,78,457,0.95,0.03,0.12,0.69
3,Lotus-Climax,231,281.0,22,31,144,1.22,0.10,0.13,0.62
4,March,524,148.0,3,16,384,0.28,0.01,0.03,0.73


In [72]:
constructor_year.describe()

,total_races,total_wins,total_podiums,total_dnfs,total_points,win_rate,podium_rate,dnf_rate,points_per_race,rolling_win_rate,rolling_podium_rate,rolling_dnf_rate,rolling_points
count,1111.0,1111.0,1111.0,1111.000000,1111.000000,1111.0,1111.0,1111.0,1111.0,1111.000000,1111.000000,1111.000000,1111.000000
mean,24.080108,1.015302,3.056706,14.067507,47.873132,0.032835,0.097993,0.62946,1.384203,0.033645,0.098920,0.630180,1.319586
std,15.088862,2.55017,5.69601,9.875985,107.622317,0.08558,0.174003,0.26897,2.7042,0.064865,0.146057,0.222738,2.349636
min,1.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
25%,8.0,0.0,0.0,5.000000,0.000000,0.0,0.0,0.5,0.0,0.000000,0.000000,0.500000,0.040000
50%,30.0,0.0,0.0,14.000000,7.000000,0.0,0.0,0.66,0.32,0.000000,0.020000,0.650000,0.430000
75%,34.0,0.0,3.0,22.000000,47.000000,0.0,0.12,0.81,1.515,0.040000,0.160000,0.760000,1.615000
max,69.0,21.0,33.0,52.000000,790.000000,1.0,1.0,1.0,18.5,0.500000,0.740000,1.000000,17.490000


In [73]:
constructor_year.head(10)

total_races  total_wins  total_podiums  \
constructor_name race_year                                           
AFM              1952                 3           0              0   
                 1953                 4           0              0   
AGS              1986                 2           0              0   
                 1987                14           0              0   
                 1988                16           0              0   
                 1989                31           0              0   
                 1990                32           0              0   
                 1991                28           0              0   
ATS              1963                17           0              0   
                 1978                32           0              0   

                            total_dnfs  total_points  win_rate  podium_rate  \
constructor_name race_year                                                    
AFM              1952                2           0.0       0.0          0.0   
                 1953                3           0.0       0.0          0.0   
AGS              1986                2           0.0       0.0          0.0   
                 1987                5           1.0       0.0          0.0   
                 1988               13           0.0       0.0          0.0   
                 1989               31           1.0       0.0          0.0   
                 1990               30           0.0       0.0          0.0   
                 1991               27           0.0       0.0          0.0   
ATS              1963               15           0.0       0.0          0.0   
                 1978               24           0.0       0.0          0.0   

                            dnf_rate  points_per_race  rolling_win_rate  \
constructor_name race_year                                                
AFM              1952           0.67              0.0               0.0   
                 1953           0.75              0.0               0.0   
AGS              1986            1.0              0.0               0.0   
                 1987           0.36             0.07               0.0   
                 1988           0.81              0.0               0.0   
                 1989            1.0             0.03               0.0   
                 1990           0.94              0.0               0.0   
                 1991           0.96              0.0               0.0   
ATS              1963           0.88              0.0               0.0   
                 1978           0.75              0.0               0.0   

                            rolling_podium_rate  rolling_dnf_rate  \
constructor_name race_year                                          
AFM              1952                       0.0              0.67   
                 1953                       0.0              0.71   
AGS              1986                       0.0              1.00   
                 1987                       0.0              0.68   
                 1988                       0.0              0.72   
                 1989                       0.0              0.79   
                 1990                       0.0              0.82   
                 1991                       0.0              0.81   
ATS              1963                       0.0              0.88   
                 1978                       0.0              0.82   

                            rolling_points  
constructor_name race_year                  
AFM              1952                 0.00  
                 1953                 0.00  
AGS              1986                 0.00  
                 1987                 0.04  
                 1988                 0.02  
                 1989                 0.03  
                 1990                 0.02  
                 1991                 0.02  
ATS              1963                 0.00  
             

In [74]:
constructor_year.shape

(1111, 13)

In [75]:
constructor_year.columns

Index(['total_races', 'total_wins', 'total_podiums', 'total_dnfs',
       'total_points', 'win_rate', 'podium_rate', 'dnf_rate',
       'points_per_race', 'rolling_win_rate', 'rolling_podium_rate',
       'rolling_dnf_rate', 'rolling_points'],
      dtype='object')

In [76]:
constructor_year = constructor_year.reset_index()
constructor_year.columns

Index(['constructor_name', 'race_year', 'total_races', 'total_wins',
       'total_podiums', 'total_dnfs', 'total_points', 'win_rate',
       'podium_rate', 'dnf_rate', 'points_per_race', 'rolling_win_rate',
       'rolling_podium_rate', 'rolling_dnf_rate', 'rolling_points'],
      dtype='object')

In [79]:
constructor_summary = constructor_year.groupby("constructor_name").agg(
    avg_roll_win = ("rolling_win_rate", "mean"),
    avg_roll_points = ("rolling_points", "mean"),
    std_roll_win = ("rolling_win_rate", "std"),
    std_roll_points = ("rolling_points", "std"),
    peak_roll_win = ("rolling_win_rate", "max"),
    peak_roll_points = ("rolling_points", "max"),
    seasons = ("race_year", "nunique")
).reset_index()

In [80]:
constructor_summary.head()

,constructor_name,avg_roll_win,avg_roll_points,std_roll_win,std_roll_points,peak_roll_win,peak_roll_points,seasons
0,AFM,0.000000,0.000000,0.000000,0.000000,0.00,0.00,2
1,AGS,0.000000,0.021667,0.000000,0.013292,0.00,0.04,6
2,ATS,0.000000,0.033750,0.000000,0.023261,0.00,0.06,8
3,Adams,0.000000,0.000000,NaN,NaN,0.00,0.00,1
4,Alfa Romeo,0.054375,1.055000,0.088541,1.171757,0.27,4.05,16


In [81]:
constructor_summary.shape

(211, 8)

In [82]:
constructor_summary = constructor_summary[constructor_summary["seasons"] >= 5]

In [83]:
constructor_summary.shape

(65, 8)

In [84]:
constructor_summary["dynasty_score"] = constructor_summary["avg_roll_win"] - constructor_summary["std_roll_win"]
constructor_summary.head()

,constructor_name,avg_roll_win,avg_roll_points,std_roll_win,std_roll_points,peak_roll_win,peak_roll_points,seasons,dynasty_score
1,AGS,0.000000,0.021667,0.000000,0.013292,0.00,0.04,6,0.000000
2,ATS,0.000000,0.033750,0.000000,0.023261,0.00,0.06,8,0.000000
4,Alfa Romeo,0.054375,1.055000,0.088541,1.171757,0.27,4.05,16,-0.034166
11,Arrows,0.000000,0.266316,0.000000,0.083281,0.00,0.39,19,0.000000
14,Aston Martin,0.000000,0.890000,0.000000,0.923298,0.00,2.20,6,0.000000


In [88]:
constructor_summary.sort_values("dynasty_score", ascending=False)[["constructor_name", "avg_roll_win", "std_roll_win", "dynasty_score"]].head(10)

,constructor_name,avg_roll_win,std_roll_win,dynasty_score
139,Mercedes,0.217647,0.101706,0.115941
69,Epperly,0.168000,0.103053,0.064947
115,Lotus-Climax,0.088571,0.030783,0.057788
73,Ferrari,0.097733,0.070530,0.027203
166,Red Bull,0.117000,0.090152,0.026848
134,McLaren,0.096727,0.076062,0.020666
88,Honda,0.031250,0.015526,0.015724
116,Lotus-Ford,0.046667,0.036697,0.009970
96,Kurtis Kraft,0.030909,0.021659,0.009251
46,Cooper-Climax,0.048889,0.039826,0.009063


In [90]:
constructor_summary.sort_values("seasons").head(15)

,constructor_name,avg_roll_win,avg_roll_points,std_roll_win,std_roll_points,peak_roll_win,peak_roll_points,seasons,dynasty_score
210,Zakspeed,0.000000,0.010000,0.000000,0.010000,0.00,0.02,5,0.000000
196,Toleman,0.000000,0.102000,0.000000,0.101094,0.00,0.22,5,0.000000
158,Prost,0.000000,0.338000,0.000000,0.164833,0.00,0.62,5,0.000000
137,McLaren-Ford,0.022000,0.886000,0.020494,0.378722,0.04,1.20,5,0.001506
131,Matra,0.000000,0.408000,0.000000,0.254303,0.00,0.64,5,0.000000
91,Jaguar,0.000000,0.222000,0.000000,0.074632,0.00,0.30,5,0.000000
87,Hesketh,0.010000,0.816000,0.007071,0.300799,0.02,1.17,5,0.002929
84,HWM,0.000000,0.022000,0.000000,0.014832,0.00,0.04,5,0.000000
37,Coloni,0.000000,0.000000,0.000000,0.000000,0.00,0.00,5,0.000000
82,Gordini,0.000000,0.410000,0.000000,0.207485,0.00,0.75,5,0.000000


In [91]:
constructor_races = (results_master_df.groupby("constructor_name")["raceId"].count()).reset_index(name = "career_races")

In [92]:
constructor_races.head()

,constructor_name,career_races
0,AFM,7
1,AGS,123
2,ATS,162
3,Adams,2
4,Alfa Romeo,451


In [93]:
constructor_summary = constructor_summary.merge(constructor_races, on = "constructor_name", how = "left")

In [95]:
constructor_summary[["constructor_name", "career_races"]].head()

,constructor_name,career_races
0,AGS,123
1,ATS,162
2,Alfa Romeo,451
3,Arrows,590
4,Aston Martin,191


In [97]:
constructor_summary = constructor_summary[constructor_summary["career_races"] >= 100]

In [98]:
constructor_summary.head()

,constructor_name,avg_roll_win,avg_roll_points,std_roll_win,std_roll_points,peak_roll_win,peak_roll_points,seasons,dynasty_score,career_races
0,AGS,0.000000,0.021667,0.000000,0.013292,0.00,0.04,6,0.000000,123
1,ATS,0.000000,0.033750,0.000000,0.023261,0.00,0.06,8,0.000000,162
2,Alfa Romeo,0.054375,1.055000,0.088541,1.171757,0.27,4.05,16,-0.034166,451
3,Arrows,0.000000,0.266316,0.000000,0.083281,0.00,0.39,19,0.000000,590
4,Aston Martin,0.000000,0.890000,0.000000,0.923298,0.00,2.20,6,0.000000,191


In [99]:
constructor_summary.shape

(49, 10)

211 raw constructors

65 constructors with ≥5 seasons

49 constructors with ≥100 race entries

In [100]:
constructor_summary.sort_values("dynasty_score", ascending=False)[["constructor_name", "career_races", "avg_roll_win", "std_roll_win", "dynasty_score"]].head(10)

,constructor_name,career_races,avg_roll_win,std_roll_win,dynasty_score
44,Mercedes,652,0.217647,0.101706,0.115941
37,Lotus-Climax,231,0.088571,0.030783,0.057788
19,Ferrari,2439,0.097733,0.070530,0.027203
50,Red Bull,788,0.117000,0.090152,0.026848
42,McLaren,1923,0.096727,0.076062,0.020666
27,Honda,152,0.031250,0.015526,0.015724
38,Lotus-Ford,128,0.046667,0.036697,0.009970
30,Kurtis Kraft,226,0.030909,0.021659,0.009251
14,Cooper-Climax,267,0.048889,0.039826,0.009063
7,Benetton,520,0.054375,0.047884,0.006491


Dynasty score = average rolling win rate minus rolling win-rate volatility; it rewards teams that win frequently and remain consistently competitive over time

Q4. How resilient are constructors to performance dips?
Observe drop + recovery pattern in rolling metrics

What you’re detecting:

Teams that bounce back quickly vs collapse

Advanced angle (this is where you stand out):
Measure drawdown and recovery time

# drop from peak
driver_year["rolling_peak"] = driver_year.groupby("driver_name")["rolling_win_rate"].cummax()
driver_year["drawdown"] = driver_year["rolling_win_rate"] - driver_year["rolling_peak"]

Actionable insight:

Teams with fast recovery = strong management/system
Long drawdowns = structural weakness

Category 3: Circuit Evolution Trends
Q5. Which circuits are becoming more dangerous or unpredictable over time?
Rolling DNF rate

What you’re detecting:

Increasing DNF trend → reliability issues, track difficulty
Stable low DNF → safe, predictable circuits

Actionable insight:

Teams prepare differently for high-risk circuits
FIA (governing body) decisions on track safety
Q6. Which circuits produce unpredictable winners?
Variability in winners over time

Advanced metric (important):
Instead of just win_rate:

Count unique winners per circuit over rolling window
circuit_variability = results_master_df.groupby(["circuitId", "race_year"])["driver_name"].nunique()

What you’re detecting:

High variability → chaotic circuits
Low variability → dominance-friendly tracks

Actionable insight:

Strategy-heavy races vs performance-heavy races
What You’re Actually Building (Important)

You are NOT building:

“driver trends dashboard”

You ARE building:

Performance evolution engine across time

That includes:

Rolling smoothing
Trend direction (slope)
Peak vs decline detection
Volatility / drawdown

Your Next Action (Non-negotiable)

Implement these 3 immediately:

Trend direction (slope proxy)
driver_year["trend"] = driver_year.groupby("driver_name")["rolling_win_rate"].diff()
Volatility (consistency over time)
driver_year["volatility"] = driver_year.groupby("driver_name")["rolling_win_rate"].rolling(5).std().reset_index(level=0, drop=True)
Peak vs decline (drawdown)
driver_year["peak"] = driver_year.groupby("driver_name")["rolling_win_rate"].cummax()
driver_year["drawdown"] = driver_year["rolling_win_rate"] - driver_year["peak"]
Bottom Line

Time-based trends are not about “seeing lines move.”

They answer:

Who is sustainably elite
Who is improving vs declining
Who is stable vs volatile
Where risk is increasing over time